# Estudo de Caso 3.2 — Calculadora de Números Adimensionais: Classificação de Regimes de Escoamento e Transporte

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 3 — Modelagem Matemática do Escoamento de Fluidos |
| **Nível** | Graduação |
| **Tempo estimado** | 45 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Cálculo algébrico direto (sem iteração temporal) |
| **Bibliografia** | Lugon Jr. et al. (2026), Cap. 3; White (2016), Cap. 5; Incropera et al. (2017), Cap. 6 |

---

## 📋 Resumo

Construímos uma calculadora interativa de números adimensionais para classificação de regimes de escoamento e transporte. A ferramenta calcula Reynolds, Froude, Prandtl, Schmidt, Péclet, Nusselt, Sherwood, Biot e Damköhler a partir de parâmetros físicos, classificando automaticamente o regime (laminar/turbulento, subcrítico/supercrítico, dominado por advecção/difusão). Quatro cenários de engenharia são analisados: escoamento em tubulação, canal aberto, trocador de calor e reator químico.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Calcular** os principais números adimensionais da mecânica dos fluidos e transferência de calor/massa
2. **Classificar** regimes de escoamento (laminar, transição, turbulento) pelo número de Reynolds
3. **Interpretar** a razão entre transporte advectivo e difusivo pelo número de Péclet
4. **Aplicar** números adimensionais na análise de similaridade entre sistemas físicos
5. **Avaliar** a importância relativa de mecanismos físicos em diferentes cenários de engenharia
6. **Construir** ferramentas computacionais reutilizáveis para análise dimensional

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## ⚙️ Seção 1: Parâmetros dos Cenários de Engenharia

### Contexto Físico

Os números adimensionais são combinações de grandezas físicas que **não dependem do sistema de unidades**. Eles permitem:

- **Classificar regimes** de escoamento e transporte
- **Estabelecer similaridade** entre modelo e protótipo
- **Simplificar equações** ao identificar termos dominantes
- **Generalizar correlações** empíricas

> 💡 **Observação:** A análise dimensional (Teorema de Buckingham π) garante que um fenômeno físico descrito por $n$ variáveis e $k$ dimensões fundamentais pode ser reduzido a $n - k$ grupos adimensionais.

### Cenários Analisados

| Cenário | Aplicação | Números Relevantes |
|:--------|:----------|:-------------------|
| Tubulação industrial | Escoamento interno | Re, Pr, Pe, Nu |
| Canal aberto | Hidráulica fluvial | Re, Fr, Pe |
| Trocador de calor | Transferência de calor | Re, Pr, Nu, Bi |
| Reator químico | Reação + transporte | Re, Sc, Pe, Da |

In [ ]:
# ============================================================
# CENÁRIO 1: ESCOAMENTO EM TUBULAÇÃO INDUSTRIAL
# ============================================================

# Água em tubulação de aço
D_tubo = 0.100            # Diâmetro interno [m]
U_tubo = 2.0              # Velocidade média [m/s]
nu_agua = 1.0e-6          # Viscosidade cinemática da água [m²/s]
k_agua = 0.60             # Condutividade térmica da água [W/(m·K)]
rho_agua = 998.0          # Massa específica da água [kg/m³]
cp_agua = 4180.0          # Calor específico da água [J/(kg·K)]
D_massa_agua = 1.0e-9     # Difusividade mássica [m²/s]

# Propriedades derivadas
alpha_agua = k_agua / (rho_agua * cp_agua)   # Difusividade térmica [m²/s]
mu_agua = nu_agua * rho_agua                  # Viscosidade dinâmica [Pa·s]

print('📏 Cenário 1 — Tubulação Industrial:')
print(f'  • D = {D_tubo*100:.0f} mm, U = {U_tubo} m/s')
print(f'  • ν = {nu_agua:.1e} m²/s, α = {alpha_agua:.2e} m²/s')
print(f'  • Fluido: Água a ~20°C')

In [ ]:
# ============================================================
# CENÁRIO 2: CANAL ABERTO (RIO)
# ============================================================

# Rio de planície
h_canal = 2.0             # Profundidade hidráulica [m]
U_canal = 0.8             # Velocidade média [m/s]
nu_rio = 1.1e-6           # Viscosidade cinemática [m²/s]
g = 9.81                  # Aceleração da gravidade [m/s²]

print('📏 Cenário 2 — Canal Aberto (Rio):')
print(f'  • h = {h_canal} m, U = {U_canal} m/s')
print(f'  • ν = {nu_rio:.1e} m²/s')

In [ ]:
# ============================================================
# CENÁRIO 3: TROCADOR DE CALOR
# ============================================================

# Óleo lubrificante em trocador
D_tc = 0.025              # Diâmetro interno do tubo [m]
U_tc = 1.5                # Velocidade do óleo [m/s]
nu_oleo = 5.0e-5          # Viscosidade cinemática do óleo [m²/s]
k_oleo = 0.15             # Condutividade térmica do óleo [W/(m·K)]
rho_oleo = 870.0          # Massa específica do óleo [kg/m³]
cp_oleo = 2000.0          # Calor específico do óleo [J/(kg·K)]
L_tc = 2.0                # Comprimento característico [m]

alpha_oleo = k_oleo / (rho_oleo * cp_oleo)   # Difusividade térmica [m²/s]

print('📏 Cenário 3 — Trocador de Calor:')
print(f'  • D = {D_tc*100:.0f} mm, U = {U_tc} m/s, L = {L_tc} m')
print(f'  • ν = {nu_oleo:.1e} m²/s, α = {alpha_oleo:.2e} m²/s')
print(f'  • Fluido: Óleo lubrificante')

In [ ]:
# ============================================================
# CENÁRIO 4: REATOR QUÍMICO
# ============================================================

# Reator CSTR com reação de degradação
D_reator = 0.500          # Diâmetro do reator [m]
U_reator = 0.3            # Velocidade característica [m/s]
nu_reator = 1.2e-6        # Viscosidade cinemática [m²/s]
D_reator_massa = 5.0e-10  # Difusividade mássica [m²/s]
k_reacao = 0.05           # Constante de reação [s⁻¹]
L_reator = 1.0            # Comprimento característico [m]

print('📏 Cenário 4 — Reator Químico:')
print(f'  • D = {D_reator*100:.0f} mm, U = {U_reator} m/s')
print(f'  • ν = {nu_reator:.1e} m²/s, D_m = {D_reator_massa:.1e} m²/s')
print(f'  • k_reação = {k_reacao} s⁻¹')

---
## 🔬 Seção 2: Verificação — Consistência dos Dados

Antes dos cálculos, verificamos a consistência física dos parâmetros.

> 📖 **Referência:** White (2016), Cap. 5, Seção 5.2.

In [ ]:
# ============================================================
# VERIFICAÇÃO — CONSISTÊNCIA DOS DADOS
# ============================================================

print('🔬 VERIFICAÇÃO DE CONSISTÊNCIA')
print('=' * 50)

# Verificar positividades
params = {
    'D_tubo': D_tubo, 'U_tubo': U_tubo, 'nu_agua': nu_agua,
    'h_canal': h_canal, 'U_canal': U_canal,
    'D_tc': D_tc, 'U_tc': U_tc, 'nu_oleo': nu_oleo,
    'D_reator': D_reator, 'U_reator': U_reator
}

todos_positivos = all(v > 0 for v in params.values())
assert todos_positivos, '❌ Parâmetros negativos ou nulos!'
print('  ✓ Todos os parâmetros são positivos')

# Verificar faixas físicas
assert 0.9 < nu_agua * 1e6 < 1.1, '❌ Viscosidade da água fora da faixa'
assert 0.1 < nu_oleo * 1e5 < 10.0, '❌ Viscosidade do óleo fora da faixa'
print('  ✓ Viscosidades dentro de faixas físicas esperadas')

# Verificar relação entre difusividades (Pr > 1 para líquidos)
Pr_verif = nu_agua / alpha_agua
print(f'  ✓ Pr(água) = {Pr_verif:.1f} (> 1, coerente para líquido)')

print(f'\n✅ Dados consistentes — prosseguir com cálculos')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Definições dos Números Adimensionais

| Número | Definição | Interpretação Física |
|:-------|:----------|:---------------------|
| Reynolds | $Re = \frac{UL}{\nu}$ | Inércia / Viscosidade |
| Froude | $Fr = \frac{U}{\sqrt{gL}}$ | Inércia / Gravidade |
| Prandtl | $Pr = \frac{\nu}{\alpha}$ | Difusão de momento / Difusão térmica |
| Schmidt | $Sc = \frac{\nu}{D_m}$ | Difusão de momento / Difusão mássica |
| Péclet térmico | $Pe_T = Re \cdot Pr = \frac{UL}{\alpha}$ | Advecção térmica / Difusão térmica |
| Péclet mássico | $Pe_M = Re \cdot Sc = \frac{UL}{D_m}$ | Advecção mássica / Difusão mássica |
| Nusselt | $Nu = \frac{hL}{k}$ | Convecção / Condução |
| Sherwood | $Sh = \frac{k_m L}{D_m}$ | Convecção mássica / Difusão |
| Biot | $Bi = \frac{hL}{k_s}$ | Resistência convectiva / Resistência condutiva |
| Damköhler | $Da = k_{reac} \cdot \tau_{res}$ | Tempo de residência / Tempo de reação |

### 3.2 Critérios de Classificação

| Número | Faixa | Classificação |
|:-------|:------|:--------------|
| $Re$ | $< 2400$ | Laminar |
| $Re$ | $2400 – 4000$ | Transição |
| $Re$ | $> 4000$ | Turbulento |
| $Fr$ | $< 0{,}95$ | Subcrítico (fluvial) |
| $Fr$ | $0{,}95 – 1{,}05$ | Crítico |
| $Fr$ | $> 1{,}05$ | Supercrítico (torrencial) |
| $Pe$ | $\ll 1$ | Dominado por difusão |
| $Pe$ | $\gg 1$ | Dominado por advecção |

> 📖 **Referência:** White (2016), Cap. 5; Incropera et al. (2017), Cap. 6; Lugon Jr. et al. (2026), Cap. 3.

---
## ⚡ Seção 4: Critério Numérico

> 💡 **Observação:** Neste estudo de caso, **não se aplica** a condição CFL ou critério de estabilidade numérica, pois não há equações diferenciais sendo resolvidas por métodos iterativos temporais. Os cálculos são **algébricos diretos** (avaliação de fórmulas). A "estabilidade" aqui é garantida pela consistência dimensional das fórmulas e pela verificação de faixas físicas dos parâmetros.

---
## 🌪️ Seção 5: Condições de Entrada (Forçamento)

Neste problema algébrico, o forçamento são os **parâmetros físicos de entrada** de cada cenário:

- Velocidade característica $U$
- Comprimento característico $L$ (diâmetro, profundidade, etc.)
- Propriedades do fluido ($\nu$, $\alpha$, $D_m$, $k$, $\rho$, $c_p$)
- Parâmetros de reação ($k_{reac}$)

> 💡 **Observação:** A escolha do comprimento característico $L$ é crucial e depende da geometria: diâmetro hidráulico para tubulações, profundidade para canais abertos, comprimento para placas.

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# FUNÇÕES DE CÁLCULO DOS NÚMEROS ADIMENSIONAIS
# ============================================================

def calcular_reynolds(U, L, nu):
    """
    Calcula o número de Reynolds.

    Parameters
    ----------
    U : float
        Velocidade característica [m/s].
    L : float
        Comprimento característico [m].
    nu : float
        Viscosidade cinemática [m²/s].

    Returns
    -------
    float
        Número de Reynolds [-].

    Notes
    -----
    Re = U * L / nu
    """
    return U * L / nu


def calcular_froude(U, L, g=9.81):
    """
    Calcula o número de Froude.

    Parameters
    ----------
    U : float
        Velocidade característica [m/s].
    L : float
        Comprimento característico (profundidade) [m].
    g : float
        Aceleração da gravidade [m/s²].

    Returns
    -------
    float
        Número de Froude [-].

    Notes
    -----
    Fr = U / sqrt(g * L)
    """
    return U / np.sqrt(g * L)


def calcular_prandtl(nu, alpha):
    """
    Calcula o número de Prandtl.

    Parameters
    ----------
    nu : float
        Viscosidade cinemática [m²/s].
    alpha : float
        Difusividade térmica [m²/s].

    Returns
    -------
    float
        Número de Prandtl [-].

    Notes
    -----
    Pr = nu / alpha
    """
    return nu / alpha


def calcular_schmidt(nu, D_m):
    """
    Calcula o número de Schmidt.

    Parameters
    ----------
    nu : float
        Viscosidade cinemática [m²/s].
    D_m : float
        Difusividade mássica [m²/s].

    Returns
    -------
    float
        Número de Schmidt [-].

    Notes
    -----
    Sc = nu / D_m
    """
    return nu / D_m


def calcular_peclet(Re, Pr_or_Sc):
    """
    Calcula o número de Péclet (térmico ou mássico).

    Parameters
    ----------
    Re : float
        Número de Reynolds [-].
    Pr_or_Sc : float
        Número de Prandtl (térmico) ou Schmidt (mássico) [-].

    Returns
    -------
    float
        Número de Péclet [-].

    Notes
    -----
    Pe = Re * Pr (térmico) ou Pe = Re * Sc (mássico)
    """
    return Re * Pr_or_Sc


def calcular_damkohler(k_reac, tau_res):
    """
    Calcula o número de Damköhler.

    Parameters
    ----------
    k_reac : float
        Constante de reação [s⁻¹].
    tau_res : float
        Tempo de residência [s].

    Returns
    -------
    float
        Número de Damköhler [-].

    Notes
    -----
    Da = k_reac * tau_res
    """
    return k_reac * tau_res


print('✅ Funções de cálculo implementadas')

In [ ]:
# ============================================================
# FUNÇÕES DE CLASSIFICAÇÃO
# ============================================================

def classificar_regime(Re):
    """
    Classifica o regime de escoamento pelo número de Reynolds.

    Parameters
    ----------
    Re : float
        Número de Reynolds [-].

    Returns
    -------
    str
        Classificação do regime.
    """
    if Re < 2400:
        return 'LAMINAR'
    elif Re < 4000:
        return 'TRANSIÇÃO'
    else:
        return 'TURBULENTO'


def classificar_froude(Fr):
    """
    Classifica o regime de canal aberto pelo número de Froude.

    Parameters
    ----------
    Fr : float
        Número de Froude [-].

    Returns
    -------
    str
        Classificação do regime.
    """
    if Fr < 0.95:
        return 'SUBCRÍTICO (fluvial)'
    elif Fr <= 1.05:
        return 'CRÍTICO'
    else:
        return 'SUPERCRÍTICO (torrencial)'


def classificar_transporte(Pe):
    """
    Classifica o regime de transporte pelo número de Péclet.

    Parameters
    ----------
    Pe : float
        Número de Péclet [-].

    Returns
    -------
    str
        Classificação do transporte.
    """
    if Pe < 0.1:
        return 'Dominado por DIFUSÃO'
    elif Pe < 10:
        return 'Regime MISTO'
    else:
        return 'Dominado por ADVECÇÃO'


print('✅ Funções de classificação implementadas')

In [ ]:
# ============================================================
# CENÁRIO 1: TUBULAÇÃO INDUSTRIAL
# ============================================================

t_inicio = time.time()

Re_tubo = calcular_reynolds(U_tubo, D_tubo, nu_agua)
Pr_agua = calcular_prandtl(nu_agua, alpha_agua)
Pe_termico = calcular_peclet(Re_tubo, Pr_agua)
Sc_agua = calcular_schmidt(nu_agua, D_massa_agua)
Pe_massico = calcular_peclet(Re_tubo, Sc_agua)

t_cpu_1 = time.time() - t_inicio

print('=' * 60)
print('  CENÁRIO 1: TUBULAÇÃO INDUSTRIAL (Água)')
print('=' * 60)
print(f'\n🔢 Números Adimensionais:')
print(f'  • Re = {Re_tubo:.0f} → {classificar_regime(Re_tubo)}')
print(f'  • Pr = {Pr_agua:.2f}')
print(f'  • Sc = {Sc_agua:.0f}')
print(f'  • Pe_térmico = {Pe_termico:.0f} → {classificar_transporte(Pe_termico)}')
print(f'  • Pe_mássico = {Pe_massico:.0f} → {classificar_transporte(Pe_massico)}')
print(f'\n💡 Interpretação:')
print(f'  • Escoamento TURBULENTO (Re >> 4000)')
print(f'  • Transporte dominado por ADVECÇÃO (Pe >> 10)')
print(f'  • Pr > 1 → camada limite térmica < camada limite hidrodinâmica')
print(f'\n⏱️ Tempo de CPU: {t_cpu_1:.6f} s')

In [ ]:
# ============================================================
# CENÁRIO 2: CANAL ABERTO (RIO)
# ============================================================

Re_canal = calcular_reynolds(U_canal, h_canal, nu_rio)
Fr_canal = calcular_froude(U_canal, h_canal)
Pe_canal = calcular_peclet(Re_canal, 7.0)  # Pr ≈ 7 para água

print('=' * 60)
print('  CENÁRIO 2: CANAL ABERTO (Rio)')
print('=' * 60)
print(f'\n🔢 Números Adimensionais:')
print(f'  • Re = {Re_canal:.0f} → {classificar_regime(Re_canal)}')
print(f'  • Fr = {Fr_canal:.3f} → {classificar_froude(Fr_canal)}')
print(f'  • Pe = {Pe_canal:.0f} → {classificar_transporte(Pe_canal)}')
print(f'\n💡 Interpretação:')
print(f'  • Escoamento TURBULENTO (Re >> 4000)')
print(f'  • Regime SUBCRÍTICO (Fr < 1) → informações propagam montante')
print(f'  • Transporte dominado por ADVECÇÃO (Pe >> 10)')

In [ ]:
# ============================================================
# CENÁRIO 3: TROCADOR DE CALOR
# ============================================================

Re_tc = calcular_reynolds(U_tc, D_tc, nu_oleo)
Pr_oleo = calcular_prandtl(nu_oleo, alpha_oleo)
Pe_tc = calcular_peclet(Re_tc, Pr_oleo)

print('=' * 60)
print('  CENÁRIO 3: TROCADOR DE CALOR (Óleo)')
print('=' * 60)
print(f'\n🔢 Números Adimensionais:')
print(f'  • Re = {Re_tc:.0f} → {classificar_regime(Re_tc)}')
print(f'  • Pr = {Pr_oleo:.0f}')
print(f'  • Pe = {Pe_tc:.0f} → {classificar_transporte(Pe_tc)}')
print(f'\n💡 Interpretação:')
print(f'  • Escoamento em {classificar_regime(Re_tc)} (Re = {Re_tc:.0f})')
print(f'  • Pr >> 1 → óleo é mau condutor térmico')
print(f'  • Pe >> 10 → advecção domina sobre difusão térmica')

In [ ]:
# ============================================================
# CENÁRIO 4: REATOR QUÍMICO
# ============================================================

Re_reator = calcular_reynolds(U_reator, D_reator, nu_reator)
Sc_reator = calcular_schmidt(nu_reator, D_reator_massa)
Pe_reator = calcular_peclet(Re_reator, Sc_reator)
tau_res_reator = L_reator / U_reator   # Tempo de residência [s]
Da_reator = calcular_damkohler(k_reacao, tau_res_reator)

print('=' * 60)
print('  CENÁRIO 4: REATOR QUÍMICO')
print('=' * 60)
print(f'\n🔢 Números Adimensionais:')
print(f'  • Re = {Re_reator:.0f} → {classificar_regime(Re_reator)}')
print(f'  • Sc = {Sc_reator:.0f}')
print(f'  • Pe = {Pe_reator:.0f} → {classificar_transporte(Pe_reator)}')
print(f'  • τ_res = {tau_res_reator:.1f} s')
print(f'  • Da = {Da_reator:.2f}')
print(f'\n💡 Interpretação:')
print(f'  • Escoamento em {classificar_regime(Re_reator)}')
print(f'  • Da = {Da_reator:.2f} → reação {"rápida" if Da_reator > 1 else "lenta"} relativa ao transporte')
print(f'  • Pe >> 10 → advecção domina sobre difusão mássica')

---
## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# GRÁFICO 1: Mapa de Classificação Re vs. Pe
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))

# Cenários como pontos
cenarios = {
    'Tubulação\n(água)': (Re_tubo, Pe_termico, 'red', 'o'),
    'Canal aberto\n(rio)': (Re_canal, Pe_canal, 'blue', 's'),
    'Trocador\n(óleo)': (Re_tc, Pe_tc, 'green', '^'),
    'Reator\nquímico': (Re_reator, Pe_reator, 'purple', 'D'),
}

# Regiões de classificação
ax.axvspan(0, 2400, alpha=0.05, color='blue', label='Re: Laminar')
ax.axvspan(2400, 4000, alpha=0.05, color='yellow', label='Re: Transição')
ax.axvspan(4000, 1e7, alpha=0.05, color='red', label='Re: Turbulento')

ax.axhspan(0, 0.1, alpha=0.05, color='green', label='Pe: Difusão')
ax.axhspan(0.1, 10, alpha=0.05, color='orange', label='Pe: Misto')
ax.axhspan(10, 1e10, alpha=0.05, color='pink', label='Pe: Advecção')

# Plotar cenários
for nome, (Re_val, Pe_val, cor, marc) in cenarios.items():
    ax.scatter(Re_val, Pe_val, c=cor, marker=marc, s=200, zorder=5,
               edgecolors='black', linewidth=1.5)
    ax.annotate(nome, xy=(Re_val, Pe_val), xytext=(15, 10),
                textcoords='offset points', fontsize=10, fontweight='bold')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Número de Reynolds (Re)', fontsize=12)
ax.set_ylabel('Número de Péclet (Pe)', fontsize=12)
ax.set_title('Mapa de Classificação: Regime Hidrodinâmico vs. Transporte',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(1e1, 1e7)
ax.set_ylim(1e-1, 1e10)

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - Todos os cenários estão na região **turbulenta** (Re > 4000) e **dominada por advecção** (Pe >> 10).
> - O trocador de calor (óleo) tem Re mais baixo devido à alta viscosidade.
> - O reator químico tem Pe mássico muito elevado (Sc >> 1), indicando que a difusão mássica é muito lenta comparada à advecção.

In [ ]:
# ============================================================
# GRÁFICO 2: Comparação de Pr e Sc
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

fluidos = ['Água', 'Óleo']
Pr_vals = [Pr_agua, Pr_oleo]
Sc_vals = [Sc_agua, Sc_reator]

x = np.arange(len(fluidos))
width = 0.35

bars1 = ax.bar(x - width/2, Pr_vals, width, label='Prandtl (Pr)',
               color='#3498db', edgecolor='black', linewidth=0.8)
bars2 = ax.bar(x + width/2, Sc_vals, width, label='Schmidt (Sc)',
               color='#e74c3c', edgecolor='black', linewidth=0.8)

# Valores em escala log
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(fluidos, fontsize=12)
ax.set_ylabel('Valor (escala log)', fontsize=12)
ax.set_title('Comparação: Prandtl vs. Schmidt', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Anotar valores
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
            f'{bar.get_height():.1f}', ha='center', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
            f'{bar.get_height():.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - Pr(água) ≈ 7: camada limite térmica e hidrodinâmica têm espessuras comparáveis.
> - Pr(óleo) ≈ 667: camada limite térmica é muito mais fina que a hidrodinâmica.
> - Sc >> 1 para todos os líquidos: difusão mássica é muito mais lenta que difusão de momento.
> - Isso explica por que mistura de espécies químicas é mais difícil que mistura térmica.

---
## 📈 Seção 8: Análise Quantitativa

In [ ]:
# ============================================================
# TABELA COMPARATIVA DOS CENÁRIOS
# ============================================================

dados_tabela = {
    'Cenário': [
        'Tubulação (água)',
        'Canal aberto (rio)',
        'Trocador (óleo)',
        'Reator químico'
    ],
    'Re': [
        f'{Re_tubo:.0f}',
        f'{Re_canal:.0f}',
        f'{Re_tc:.0f}',
        f'{Re_reator:.0f}'
    ],
    'Regime': [
        classificar_regime(Re_tubo),
        classificar_regime(Re_canal),
        classificar_regime(Re_tc),
        classificar_regime(Re_reator)
    ],
    'Pr ou Sc': [
        f'Pr = {Pr_agua:.1f}',
        f'Pr ≈ 7',
        f'Pr = {Pr_oleo:.0f}',
        f'Sc = {Sc_reator:.0f}'
    ],
    'Pe': [
        f'{Pe_termico:.0f}',
        f'{Pe_canal:.0f}',
        f'{Pe_tc:.0f}',
        f'{Pe_reator:.0f}'
    ],
    'Transporte': [
        classificar_transporte(Pe_termico),
        classificar_transporte(Pe_canal),
        classificar_transporte(Pe_tc),
        classificar_transporte(Pe_reator)
    ],
    'Fr': [
        '—',
        f'{Fr_canal:.3f}',
        '—',
        '—'
    ],
    'Da': [
        '—',
        '—',
        '—',
        f'{Da_reator:.2f}'
    ]
}

df_comp = pd.DataFrame(dados_tabela)
display(Markdown('### Tabela Comparativa dos Cenários'))
display(Markdown(df_comp.to_markdown(index=False)))

---
## 🔍 Seção 9: Análise de Sensibilidade

Analisamos como a variação da velocidade afeta a classificação dos regimes.

In [ ]:
# ============================================================
# SENSIBILIDADE — Efeito da velocidade no Reynolds
# ============================================================

U_range = np.linspace(0.01, 10.0, 100)

Re_tubo_range = [calcular_reynolds(U, D_tubo, nu_agua) for U in U_range]
Re_canal_range = [calcular_reynolds(U, h_canal, nu_rio) for U in U_range]
Fr_canal_range = [calcular_froude(U, h_canal) for U in U_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Painel 1: Re vs. U
ax1.semilogy(U_range, Re_tubo_range, 'b-', linewidth=2, label=f'Tubulação (D={D_tubo*100:.0f} mm)')
ax1.semilogy(U_range, Re_canal_range, 'r-', linewidth=2, label=f'Canal (h={h_canal} m)')
ax1.axhline(2400, color='orange', linestyle='--', linewidth=1.5, label='Re = 2400 (laminar→transição)')
ax1.axhline(4000, color='red', linestyle='--', linewidth=1.5, label='Re = 4000 (transição→turbulento)')
ax1.axvline(U_tubo, color='blue', linestyle=':', alpha=0.5, label=f'U_tubo = {U_tubo} m/s')
ax1.axvline(U_canal, color='red', linestyle=':', alpha=0.5, label=f'U_canal = {U_canal} m/s')
ax1.set_xlabel('Velocidade U [m/s]', fontsize=12)
ax1.set_ylabel('Número de Reynolds (Re)', fontsize=12)
ax1.set_title('(a) Reynolds vs. Velocidade', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, which='both')

# Painel 2: Fr vs. U (canal aberto)
ax2.plot(U_range, Fr_canal_range, 'g-', linewidth=2)
ax2.axhline(1.0, color='red', linestyle='--', linewidth=2, label='Fr = 1 (crítico)')
ax2.axvline(U_canal, color='green', linestyle=':', alpha=0.5, label=f'U_canal = {U_canal} m/s')
ax2.fill_between(U_range, 0, Fr_canal_range, where=np.array(Fr_canal_range) < 1,
                 alpha=0.1, color='blue', label='Subcrítico')
ax2.fill_between(U_range, 0, Fr_canal_range, where=np.array(Fr_canal_range) > 1,
                 alpha=0.1, color='red', label='Supercrítico')
ax2.set_xlabel('Velocidade U [m/s]', fontsize=12)
ax2.set_ylabel('Número de Froude (Fr)', fontsize=12)
ax2.set_title('(b) Froude vs. Velocidade (Canal)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 3)

plt.tight_layout()
plt.show()

# Encontrar velocidades críticas
U_crit_laminar_tubo = 2400 * nu_agua / D_tubo
U_crit_turb_tubo = 4000 * nu_agua / D_tubo
U_crit_Fr1 = np.sqrt(g * h_canal)

print(f'\n📊 Velocidades críticas:')
print(f'  • Tubulação: laminar→transição em U = {U_crit_laminar_tubo:.3f} m/s')
print(f'  • Tubulação: transição→turbulento em U = {U_crit_turb_tubo:.3f} m/s')
print(f'  • Canal: Fr = 1 (crítico) em U = {U_crit_Fr1:.2f} m/s')

> 💡 **Observação:**
> - Para a tubulação de 100 mm, o regime laminar só ocorre para U < 0,024 m/s (muito lento).
> - Para o canal com h = 2 m, o regime crítico ocorre em U ≈ 4,4 m/s.
> - Na prática, a maioria dos escoamentos de engenharia é turbulenta.
> - O número de Froude é o parâmetro dominante em canais abertos (superfície livre).

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Reynolds e a transição de regime:** O número de Reynolds representa a razão entre forças de inércia e forças viscosas. Para Re < 2400, a viscosidade domina e o escoamento é laminar (camadas paralelas). Para Re > 4000, a inércia domina e surgem vórtices caóticos (turbulência).

2. **Péclet e o transporte:** O número de Péclet indica se o transporte de calor/massa é dominado por advecção (movimento do fluido) ou difusão (movimento molecular). Pe >> 1 significa que o transporte advectivo é muito mais rápido que o difusivo.

3. **Froude e superfície livre:** Em canais abertos, o número de Froude determina se perturbações podem se propagar montante (subcrítico, Fr < 1) ou apenas jusante (supercrítico, Fr > 1). Isso é fundamental para projeto de vertedores, ressalto hidráulico e controle de cheias.

4. **Damköhler e reação:** O número de Damköhler compara o tempo de residência do fluido no reator com o tempo característico da reação. Da >> 1 significa que a reação é rápida e o conversão é limitada pelo transporte. Da << 1 significa que o transporte é rápido e a conversão é limitada pela cinética.

### Limitações do Modelo

- **Propriedades constantes:** Assume propriedades do fluido constantes. Na realidade, variam com temperatura e pressão.
- **Comprimento característico único:** A escolha de L é simplificada. Geometrias complexas podem ter múltiplos comprimentos relevantes.
- **Classificação por faixas fixas:** Os limites Re = 2400 e Re = 4000 são aproximados e dependem da rugosidade, perturbações de entrada, etc.
- **Sem efeitos de escala:** A análise dimensional garante similaridade apenas se todos os números relevantes forem iguais (o que raramente é possível simultaneamente).

### Aplicações Práticas

- **Engenharia hidráulica:** Classificação de rios (subcrítico/supercrítico), projeto de vertedores e canais.
- **Engenharia térmica:** Seleção de correlações de transferência de calor (Nu = f(Re, Pr)).
- **Engenharia química:** Projeto de reatores (Da), colunas de absorção (Sc, Sh).
- **Engenharia ambiental:** Modelagem de dispersão de poluentes (Pe), reatores de tratamento (Da).

### Conexão com a Teoria

> 📖 **Referência:** Os números adimensionais e a análise dimensional são apresentados em White (2016), Cap. 5. As correlações de transferência de calor seguem Incropera et al. (2017), Cap. 6. A aplicação a problemas ambientais segue Lugon Jr. et al. (2026), Cap. 3, Seção 3.4.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 3](../notebooks/03_Modelagem_Matematica.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 3.2

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>